In [2]:
import pandas as pd
import random
import os
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
from datasets import load_dataset

# 1. INITIALIZE TOKENIZER (The "LLM-Specific" link)
# We use the XLM-RoBERTa tokenizer as it supports Arabic and English (Code-Switching)
model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2. DATA LOADING & ENCODING RECOVERY
def clean_and_load(file_path, is_github=True):
    if not os.path.exists(file_path): return pd.DataFrame()
    try:
        # Fixes the 'jumbled symbols' encoding issues
        df = pd.read_csv(file_path, encoding='utf-8-sig')
    except:
        df = pd.read_csv(file_path, encoding='cp1256')

    # Standardize columns to [text, sentiment]
    if is_github:
        df = df.iloc[:, [0, 1]]
    else:
        df = df[['review_description', 'rating']] if 'review_description' in df.columns else df.iloc[:, [0, 1]]
    df.columns = ['text', 'sentiment']
    return df.dropna(subset=['text'])

# 3. GULF CODE-SWITCHING INJECTION (Domain Adaptation)
dubai_cs_dict = {"جيد": "good", "سيء": "bad", "سريع": "fast", "خدمة": "service", "ممتاز": "amazing"}

def inject_cs(text):
    """Simulates Dubai/Gulf dialect by mixing English words into Arabic text."""
    words = str(text).split()
    new_text = [dubai_cs_dict.get(w, w) if random.random() < 0.3 else w for w in words]
    return " ".join(new_text)

# --- EXECUTION FLOW ---

# Load and Merge all sources
res_df = clean_and_load('/content/sample_data/RES.csv', is_github=True)
htl_df = clean_and_load('/content/sample_data/HTL.csv', is_github=True)
kaggle_df = clean_and_load('/content/sample_data/CompanyReviews.csv', is_github=False)
master_df = pd.concat([res_df, htl_df, kaggle_df], ignore_index=True)

# Preprocessing Step A: Label Binarization (0=Neg, 1=Pos)
master_df['sentiment'] = master_df['sentiment'].apply(lambda x: 1 if str(x) in ['1', 'pos', 'positive'] or (isinstance(x, (int, float)) and x >= 3) else 0)

# Preprocessing Step B: Dialectal Augmentation (Injection)
master_df['text'] = master_df.apply(lambda x: inject_cs(x['text']) if random.random() > 0.5 else x['text'], axis=1)

# Preprocessing Step C: Data Splitting (80/10/10)
train_df, temp_df = train_test_split(master_df, test_size=0.2, random_state=42, stratify=master_df['sentiment'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['sentiment'])

# Save processed CSVs for other sub-teams
train_df.to_csv("train.csv", index=False, encoding='utf-8-sig')
val_df.to_csv("val.csv", index=False, encoding='utf-8-sig')
test_df.to_csv("test.csv", index=False, encoding='utf-8-sig')

# Preprocessing Step D: LLM-Specific Tokenization
def tokenize_function(examples):
    # This creates input_ids and attention_masks for the LLM
    result = tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)
    result["labels"] = examples["sentiment"]
    return result

# Load into HuggingFace format and tokenize
dataset = load_dataset('csv', data_files={'train': 'train.csv', 'val': 'val.csv', 'test': 'test.csv'})
tokenized_datasets = dataset.map(tokenize_function, batched=True)

print("✅ LLM-specific Data Preprocessing is complete.")
print(f"Final Cleaned Dataset: {len(master_df)} rows")
print(f"Tokenized Format Example: {tokenized_datasets['train'][0].keys()}")

Generating train split: 0 examples [00:00, ? examples/s]

Generating val split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/53269 [00:00<?, ? examples/s]

Map:   0%|          | 0/6659 [00:00<?, ? examples/s]

Map:   0%|          | 0/6659 [00:00<?, ? examples/s]

✅ LLM-specific Data Preprocessing is complete.
Final Cleaned Dataset: 66587 rows
Tokenized Format Example: dict_keys(['text', 'sentiment', 'input_ids', 'attention_mask', 'labels'])
